# Lezione 11 — Casi d'uso reali e chiusura del corso

## Obiettivi di apprendimento

- Collegare ciascun algoritmo del corso (VQE, QAOA, QRL) a un'applicazione reale
- Capire perché la validazione dell'hardware (supremazia quantistica) è un requisito preliminare a qualsiasi applicazione

## Acronimi usati in questa lezione

- **QAOA**: Algoritmo Quantistico Approssimato di Ottimizzazione
- **QRL**: apprendimento per rinforzo quantistico (Quantum Reinforcement Learning)
- **TFQ**: TensorFlow Quantum
- **VQE**: Risolutore Variazionale Quantistico di Autovalori (Variational Quantum Eigensolver)

## Perché è importante

Hai percorso l'intero corso: dai qubit e dalle porte quantistiche fino al VQE avanzato. È il momento di fare un passo indietro e rispondere alla domanda che probabilmente ti sei posto nella prima lezione: a cosa serve tutto questo nella vita reale? Questa lezione di chiusura raccoglie quattro applicazioni concrete di ciò che hai imparato, tutte costruite sullo stesso materiale studiato durante il corso, così da concludere con una visione chiara di dove puoi portare queste conoscenze.

## Chimica quantistica: calcolare l'energia di una molecola reale

L'esempio più citato dell'applicazione di VQE è il calcolo della **curva di dissociazione** di una molecola di idrogeno (H₂): l'energia del sistema in funzione della distanza tra i due atomi. È lo stesso problema che hai risolto con hamiltoniani semplici, ma ora quell'operatore deriva dalla fisica reale della molecola (tramite la trasformazione di Jordan-Wigner, che converte un problema di elettroni in un problema di qubit) invece di essere inventato da te. Le aziende farmaceutiche e dei materiali seguono da vicino questo tipo di calcolo perché promette di progettare nuovi farmaci e materiali simulandone il comportamento quantistico reale, qualcosa che un computer classico può solo approssimare per molecole molto piccole.

## Ottimizzazione combinatoria: da Max-Cut ai problemi logistici

Il problema Max-Cut che hai risolto con QAOA è un modello semplificato di problemi molto più applicativi: distribuire compiti tra server per minimizzare la comunicazione tra loro, raggruppare clienti per affinità o progettare reti che riducano le interferenze. La stessa struttura matematica (dividere un insieme di elementi collegati tra loro in gruppi che ottimizzino una certa metrica) si ripete in logistica, telecomunicazioni e progettazione di reti elettriche.

## Controllo e processo decisionale: da CartPole alla robotica

L'agente che ha imparato a mantenere in equilibrio un'asta in CartPole usa lo stesso principio di cui avresti bisogno per controllare un robot, gestire un portafoglio di investimenti o decidere quando caricare una batteria: osserva uno stato, sceglie un'azione e riceve una ricompensa dalla quale impara. Sostituire la rete neurale classica con un circuito quantistico resta un campo di ricerca attivo, ma dimostra già che il QRL può trovare politiche di controllo competitive in problemi di dimensioni ridotte.

## Validare l'hardware: la supremazia quantistica come controllo di qualità

Ricorda la terza lezione: l'esperimento di supremazia quantistica di Sycamore non risolveva di per sé un problema utile, ma dimostrava che l'hardware quantistico si comporta come prevede la teoria. Questo stesso tipo di verifica (confrontare il comportamento reale di un chip con ciò che anticipa il modello teorico) è il primo passo obbligatorio prima di fidarsi di una qualsiasi delle applicazioni precedenti: a poco serve un circuito di VQE o QAOA ben progettato se l'hardware che lo esegue non è affidabile.

## Esempio di codice commentato: estensione facoltativa

Il seguente esempio calcola la curva di dissociazione dell'H₂ con VQE, usando la libreria `openfermion` per ottenere l'hamiltoniano molecolare reale. Installare `openfermion` e `openfermionpyscf` in Colab richiede diversi minuti, perché compilano la libreria di chimica quantistica `pyscf`, quindi questo blocco resta come **estensione facoltativa**: non serve eseguirlo per completare il corso, ma è incluso commentato per mostrarti come VQE passi da un esempio di lezione a un caso d'uso reale.

!pip install -q openfermion openfermionpyscf (richiede diversi minuti, facoltativo)

In [ ]:
# OPCIONAL - no obligatorio para el curso, instalación lenta (openfermion + openfermionpyscf compilan pyscf)
# Instalación (varios minutos): !pip install openfermion openfermionpyscf
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import openfermion as of
from openfermionpyscf import generate_molecular_hamiltonian
from scipy.sparse import linalg

def capa_ansatz(circuito, qubits, parametros):
    for i, qubit in enumerate(qubits):
        circuito += cirq.ry(parametros[3 * i]).on(qubit)
        circuito += cirq.rz(parametros[3 * i + 1]).on(qubit)
        circuito += cirq.ry(parametros[3 * i + 2]).on(qubit)
    for i in range(len(qubits) - 1):
        circuito += cirq.CNOT(qubits[i], qubits[i + 1])
    return circuito

def construir_vqe(qubits, capas, hamiltoniano):
    num_parametros = capas * 3 * len(qubits)
    parametros = sympy.symbols(f"vqe0:{num_parametros}")
    circuito = cirq.Circuit()
    for _ in range(capas):
        circuito = capa_ansatz(circuito, qubits, parametros)
    entradas = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
    salida = tfq.layers.PQC(circuito, hamiltoniano, differentiator=tfq.differentiators.Adjoint())(entradas)
    return tf.keras.models.Model(inputs=entradas, outputs=salida)

distancias = [0.3, 0.6, 0.9, 1.2, 1.5, 2.0]  # distancia entre los dos átomos de hidrógeno, en angstroms
energias_reales, energias_vqe = [], []

for distancia in distancias:
    geometria = [("H", (0.0, 0.0, 0.0)), ("H", (0.0, 0.0, distancia))]
    hamiltoniano_molecular = generate_molecular_hamiltonian(geometria, "sto-3g", multiplicity=1, charge=0)
    num_qubits = of.count_qubits(hamiltoniano_molecular)
    qubits = [cirq.GridQubit(0, i) for i in range(num_qubits)]

    operador_jw = of.transforms.jordan_wigner(hamiltoniano_molecular)
    matriz_dispersa = of.get_sparse_operator(operador_jw)
    autovalor_real, _ = linalg.eigsh(matriz_dispersa, k=1, which="SA")
    hamiltoniano_cirq = of.transforms.qubit_operator_to_pauli_sum(operador_jw, qubits=qubits)

    modelo = construir_vqe(qubits, capas=2, hamiltoniano=hamiltoniano_cirq)
    entrada_vacia = tfq.convert_to_tensor([cirq.Circuit()])
    optimizador = tf.keras.optimizers.Adam(learning_rate=0.1)

    energia_actual, anterior = None, np.inf
    for _ in range(200):
        with tf.GradientTape() as cinta:
            energia_actual = modelo(entrada_vacia)
        gradientes = cinta.gradient(energia_actual, modelo.trainable_variables)
        optimizador.apply_gradients(zip(gradientes, modelo.trainable_variables))
        energia_actual = energia_actual.numpy()[0][0]
        if abs(energia_actual - anterior) < 1e-5:
            break
        anterior = energia_actual

    energias_vqe.append(energia_actual)
    energias_reales.append(autovalor_real[0])

plt.plot(distancias, energias_vqe, "o-", label="VQE (TensorFlow Quantum)")
plt.plot(distancias, energias_reales, "--", label="Energía exacta")
plt.xlabel("Distancia entre átomos (angstroms)")
plt.ylabel("Energía (Hartree)")
plt.title("Curva de disociación del H₂ con VQE")
plt.legend()
plt.show()

La curva risultante ha la forma di una «U»: l'energia scende all'inizio (gli atomi si attraggono), tocca un minimo (la distanza di legame reale della molecola) e risale quando li separi troppo (il legame si rompe). Verificare che la curva di VQE si sovrapponga all'energia esatta è la validazione finale che l'algoritmo funziona su un sistema fisico reale, non solo su esempi giocattolo.

## Riepilogo dei punti chiave

- Il VQE si applica alla chimica quantistica reale calcolando curve di dissociazione molecolare, con l'hamiltoniano generato a partire dalla geometria reale di una molecola
- Il QAOA su Max-Cut modella problemi di ottimizzazione logistica, di reti e di distribuzione delle risorse
- Il QRL, addestrato in ambienti di controllo come CartPole, punta verso applicazioni di robotica e di processo decisionale sequenziale
- La verifica dell'hardware tramite supremazia quantistica è il passo preliminare indispensabile per fidarsi di una qualsiasi delle applicazioni precedenti
- Tutte queste applicazioni partono dagli stessi elementi che hai imparato nel corso: qubit, circuiti parametrizzati, hamiltoniani e ottimizzatori classici o quantistici

## Fine del corso

Hai iniziato questo corso senza conoscenze pregresse e ora comprendi qubit, porte, entanglement, supremazia quantistica, TensorFlow Quantum, classificatori quantistici, apprendimento per rinforzo quantistico, QAOA e VQE. Hai ormai gli strumenti per leggere un articolo di ricerca sul calcolo quantistico attuale e capire di cosa parla.

**Continua a praticare, scegli un caso d'uso che ti interessi davvero e costruisci il tuo progetto**: è il modo migliore per consolidare tutto ciò che hai imparato.

Complimenti per essere arrivato fino alla fine del corso!